# LP3 Practical 8 – Uber Fare Prediction
**Aim:** Preprocess the Uber dataset, identify outliers, study correlation, implement Linear Regression and Random Forest Regression, and compare R², RMSE and MAE.

**Dataset:** `uber.csv`

Run cells from top to bottom. The notebook expects the `Datasets` folder to be beside the `Notebooks` folder.

## Cell 1 – Import libraries

If a library is missing, install it in a separate cell using ` %pip install library-name ` and restart the kernel if required.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


## Cell 2 – Load the dataset

In [ ]:
from pathlib import Path

def get_dataset_path(filename):
    candidates = [
        Path.cwd() / "Datasets" / filename,
        Path.cwd().parent / "Datasets" / filename
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Keep the Datasets folder beside the Notebooks folder."
    )

DATA_PATH = get_dataset_path("uber.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
print(df.info())


## Cell 3 – Preprocess data and remove invalid records

In [ ]:
# Remove an unnecessary index column if it exists.
df = df.drop(columns=["Unnamed: 0", "key"], errors="ignore")

# Convert pickup time into datetime.
df["pickup_datetime"] = pd.to_datetime(
    df["pickup_datetime"], errors="coerce", utc=True
)

# Remove missing values and duplicate rows.
df = df.dropna().drop_duplicates()

# Keep realistic fare amounts and valid passenger counts.
df = df[
    (df["fare_amount"] > 0) &
    (df["fare_amount"] <= 100) &
    (df["passenger_count"].between(1, 6))
].copy()

# Keep valid coordinate ranges and restrict to the New York area
# to remove clearly invalid/outlier coordinates in this dataset.
df = df[
    df["pickup_latitude"].between(40, 42) &
    df["dropoff_latitude"].between(40, 42) &
    df["pickup_longitude"].between(-75, -72) &
    df["dropoff_longitude"].between(-75, -72)
].copy()

print("Shape after cleaning:", df.shape)
display(df.head())


## Cell 4 – Feature engineering (distance and time features)

In [ ]:
# Haversine formula calculates approximate distance between two GPS points.
def haversine_distance(lat1, lon1, lat2, lon2):
    earth_radius_km = 6371
    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 2 * earth_radius_km * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine_distance(
    df["pickup_latitude"],
    df["pickup_longitude"],
    df["dropoff_latitude"],
    df["dropoff_longitude"]
)

df["hour"] = df["pickup_datetime"].dt.hour
df["day"] = df["pickup_datetime"].dt.day
df["month"] = df["pickup_datetime"].dt.month
df["weekday"] = df["pickup_datetime"].dt.weekday

# Remove trips with almost zero distance.
df = df[df["distance_km"] > 0.1].copy()

features = [
    "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude",
    "passenger_count", "distance_km",
    "hour", "day", "month", "weekday"
]
target = "fare_amount"

print("Features:", features)
print("Rows available:", len(df))


## Cell 5 – Visualize outliers and correlation

In [ ]:
plt.figure(figsize=(7, 4))
plt.boxplot(df["fare_amount"])
plt.title("Box Plot of Uber Fare Amount")
plt.ylabel("Fare Amount")
plt.show()

plt.figure(figsize=(10, 7))
corr = df[features + [target]].corr(numeric_only=True)
plt.imshow(corr, cmap="coolwarm", aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


## Cell 6 – Split data into training and testing sets

In [ ]:
# Use a reproducible sample to keep Random Forest training manageable.
model_df = df.sample(n=min(30000, len(df)), random_state=42)

X = model_df[features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## Cell 7 – Train Linear Regression

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)
print("Linear Regression model trained.")


## Cell 8 – Train Random Forest Regression

In [ ]:
random_forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
random_forest_model.fit(X_train, y_train)

rf_predictions = random_forest_model.predict(X_test)
print("Random Forest model trained.")


## Cell 9 – Compare model performance

In [ ]:
def regression_metrics(actual, predicted):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)
    return rmse, mae, r2

lr_rmse, lr_mae, lr_r2 = regression_metrics(y_test, linear_predictions)
rf_rmse, rf_mae, rf_r2 = regression_metrics(y_test, rf_predictions)

results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest Regression"],
    "RMSE": [lr_rmse, rf_rmse],
    "MAE": [lr_mae, rf_mae],
    "R2 Score": [lr_r2, rf_r2]
})

display(results)


## Cell 10 – Actual vs predicted values

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, rf_predictions, alpha=0.3)
plt.xlabel("Actual Fare")
plt.ylabel("Predicted Fare")
plt.title("Random Forest: Actual vs Predicted Fare")
plt.show()

print("Sample actual fares:", y_test.head().round(2).tolist())
print("Sample predicted fares:", np.round(rf_predictions[:5], 2).tolist())


## Sample output / observation
- The notebook prints the cleaned dataset size and train/test sizes.
- It displays RMSE, MAE and R² for both regression models.
- Lower RMSE/MAE and higher R² indicate better regression performance.
- Exact metric values may vary if the sample size or preprocessing is changed.